## SWE - JAX

JAX takes a different angle from NumPy and Numba. You write
the kernel as a **pure function over whole arrays**, decorate it with
`@jax.jit`, and the XLA compiler produces a single GPU program for the
entire time loop, running every step on the device without returning to Python.

### Table of Contents

1. [Imports and reference setup](#sec1)
2. [The step as a pure function](#sec2)
3. [Fuse the time loop with `lax.scan`](#sec3)
4. [Acceptance gate](#sec4)
5. [Limitation: shape-rigidity](#sec5)
6. [Fixed cost: when does N start to matter?](#sec6)
7. [Cache residency](#sec7)

### <a id="sec1"></a>1. Imports and reference setup

We re-import `swe_core` for the float64 reference field, then load JAX.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import swe_core

# Shared problem parameters
N, N_STEPS = swe_core.canonical_size()
L       = 10.0
H0      = 1.0
AMP     = 0.1
SIG     = 0.5
CFL     = 0.4
G       = 9.81
dx      = L / N
DT      = swe_core.fixed_dt(H0 + AMP, dx, cfl=CFL, g=G)

# Float64 NumPy reference for validation
h_ref, _ = swe_core.solve_numpy(N, N_STEPS)

import jax
import jax.numpy as jnp
from jax import lax

### <a id="sec2"></a>2. The step as a pure function

Let's look at the same operation shape as notebook 08, expressed in JAX:

- **Read**. `h[:-1]`, `h[1:]` for interface states, same as NumPy-style slicing.
- **Compute**. Rusanov formula unchanged: `F_h = 0.5*(huL+huR) - 0.5*a*(hR-hL)`.
- **Update**. JAX arrays are immutable, so we produce a new array with `h.at[1:-1].set(...)` instead of `h[1:-1] = ...`.

The function signature `(state, _) -> ((new_state), None)` is the shape `lax.scan` expects in the next section.

---

**Quick Docs**

- `jax.jit(fn)`: just-in-time compile a Python function on its first call
  with a given input shape/dtype. Subsequent calls reuse the compiled code.
- `jax.numpy` mirrors NumPy with immutable arrays.
- `jnp.maximum`, `jnp.abs`, `jnp.sqrt`: elementwise math.
- `jnp.array.at[idx].set(value)`: functional update for an immutable array, returns a *new* array with the slot set.
- `jax.lax.scan(body, init, xs)`: a structured loop the compiler can fuse.
  Equivalent to a Python `for x in xs: carry, y = body(carry, x)`, but
  inside a single XLA program.
- `arr.block_until_ready()`: synchronise; required before stopping any GPU
  timer.

In [ ]:
@jax.jit
def step_jax(state, _):
    '''One Rusanov-flux step, fully functional. Returns (new_state, None).

    TODO: fill in the functional Rusanov step below.

    The shape of the answer:
      - h, hu = state  (each shape (N+2,))
      - apply reflective BCs via .at[0].set(...) / .at[-1].set(...)
      - compute interface states at i+1/2 for i = 0..N
      - h_safe = max(h, DRY); u = hu / h_safe; c = sqrt(g h_safe)
      - a = max(|uL|+cL, |uR|+cR)
      - F_h  = 0.5(huL+huR) - 0.5 a (hR - hL)
      - F_hu = 0.5(huL*uL + 0.5 g hL^2 + huR*uR + 0.5 g hR^2) - 0.5 a (huR - huL)
      - h_new  = h.at[1:-1].set(h[1:-1]  - (DT/dx)(F_h[1:]  - F_h[:-1]))
      - hu_new = hu.at[1:-1].set(hu[1:-1] - (DT/dx)(F_hu[1:] - F_hu[:-1]))

    Return ((h_new, hu_new), None) for use with lax.scan.
    '''
    h, hu = state
    DRY = jnp.float32(swe_core.DRY_TOL)
    # TODO: implement the Rusanov step (see swe_core.step_numpy for the
    # equivalent NumPy version; the JAX version differs only in functional
    # updates with .at[idx].set(...)).
    h_new = ...
    hu_new = ...
    return (h_new, hu_new), None

### <a id="sec3"></a>3. Fuse the time loop with `lax.scan`

`for _ in range(N_STEPS): state = step_jax(state, ...)` would
*work*, but each call would round-trip through the Python interpreter
between steps, preventing XLA from fusing them.

`jax.lax.scan(body, init, xs)` collapses the entire loop into one device
program. The cost is that `step_jax` must already be jit-able, and
the scan body must accept a `(state, x)` signature.

---

**Quick Docs**

- `jax.lax.scan(body, init, xs)`: walks the leading axis of `xs`,
  threading `state` through `body`. We do not need the per-step `x` value
  (we use a fixed `DT`), so we pass `jnp.arange(N_STEPS)` and ignore it
  inside the body.
- `jax.tree_util.tree_map(lambda a: a.block_until_ready(), pytree)`:
  synchronise every leaf in a pytree of arrays. We use it as the sync
  before stopping a timer.

In [ ]:
setup_ic = swe_core.by_size(lambda n: tuple(
    jnp.asarray(a, jnp.float32)
    for a in swe_core.bump_ic(n, L=L, h0=H0, amplitude=AMP, sigma=SIG)))


def run_jax(N_local: int):
    '''One full N_STEPS simulation at a given grid size, returns (h, hu) on device.

    TODO: time-step the prepared device state `setup_ic(N_local)` through
    N_STEPS with jax.lax.scan, then return the two arrays.
    Hint: one jax.lax.scan call replaces the whole Python `for` loop and lets
    XLA fuse every step into a single device program. Synchronise with
    jax.tree_util.tree_map(lambda a: a.block_until_ready(), final) before
    returning, or the timer stops before the GPU does. Leave the result on the
    device; a host copy inside the call would be timed too.
    '''
    final, _ = ...
    return ...

### <a id="sec4"></a>4. Acceptance gate

Cold time captures the first call (XLA compile + execute). Warm time is
the steady-state, after a couple of warmups. We also check the JAX
float32 result against the NumPy float64 reference.

---

**Quick Docs**

- A float32 trajectory accumulates round-off of order `n·ε` over the run,
  which the cell below reports. The acceptance tolerance is `1e-4` here.
- `swe_core.report_and_verify(warm, diff, tol, ...)`: verify the result is
  within the tolerance and emit one timing and acceptance record.

In [ ]:
# Cold capture.
t0 = time.perf_counter()
h_f, hu_f = run_jax(N)
cold_s = time.perf_counter() - t0

# Warm timing
warm = swe_core.timed_run(run_jax, N, warmup=2, repeats=5, label='09_jax')

# Acceptance.
diff = swe_core.max_diff(h_ref, np.asarray(h_f))
swe_core.report_and_verify(warm, diff, tol=1e-4, cold_s=cold_s,
                           n=N, steps=N_STEPS, cold_note='incl. XLA compile')

swe_core.save_timing(
    warm, grid_str=f'N={N}', tool='jax', hardware='gpu',
    dtype='float32', steps=N_STEPS,
    cold_s=cold_s, max_diff_vs_numpy=diff,
)

### <a id="sec5"></a>5. Limitation: shape-rigidity

The XLA compile happens **per input shape**.

The chart runs each shape once and splits that first (cold) call into the
one-time compile and the recurring execution.

What makes the warm runs faster, and why does a new `N` require repaying the compilation cost?

---

**Quick Docs**

- JAX requires the same shape and dtype for a warm run.

In [ ]:
# Fresh shapes: each pays one XLA compile on its cold call.
shape_data = []
for N_local in [n for n, _ in swe_core.sweep_points()]:
    t0 = time.perf_counter()
    run_jax(N_local)
    cold = time.perf_counter() - t0
    warm = swe_core.timed_run(run_jax, N_local, warmup=2, repeats=3)['median_s']
    shape_data.append((N_local, cold, warm))
    print(f'  N={N_local:>10,}:  cold {cold*1e3:8.1f} ms   warm {warm*1e3:8.1f} ms   '
          f'compile ~ cold-warm = {(cold - warm)*1e3:6.0f} ms')

swe_core.plot_compile_share([f'N={n:,}' for n, _, _ in shape_data],
                            [c for _, c, _ in shape_data],
                            [w for _, _, w in shape_data],
                            'Breakdown of runtime with growing N')

### <a id="sec6"></a>6. Fixed cost: when does N start to matter?

Looking at the warm bars in Sec. 5: at the smallest sweep size the run is
dominated by a fixed per-step cost, the kernel launch and scan-step dispatch in
the XLA program. As `N` grows that cost is amortised and the time becomes
proportional to the cells.

We can explain the curve with a two-term model:

$$t_\mathrm{warm}(N) \approx n_\mathrm{steps} \cdot (t_0 + N/B)$$

where $t_0$ is the fixed per-step overhead and $B$ is the streaming rate.
Below the crossover $N^* = t_0 \cdot B$ the linear term is invisible; above
it, doubling `N` doubles the time. On GPUs with a large L2 cache there is a third
regime in between (Sec. 7).

**Q:** at what `N` do you expect the linear term to take over on this GPU?


In [ ]:
# Warm sweep over a wide range of N.
SWEEP_N = [n for n, _ in swe_core.sweep_points()]

sweep_t = []
for N_local in SWEEP_N:
    r = swe_core.timed_run(run_jax, N_local, warmup=2, repeats=3)
    sweep_t.append(r['median_s'])
    print(f'  N={N_local:>9,}:  warm {r["median_s"]*1e3:8.1f} ms   '
          f'{N_local * N_STEPS / r["median_s"] / 1e6:8.0f} Mcells/s')

Ns = np.array(SWEEP_N, dtype=float)
ts = np.array(sweep_t)

# Two-term fit: t0 from the small-N plateau, slope from the two largest sizes.
t0_step = ts[0] / N_STEPS
slope   = (ts[-1] - ts[-2]) / (Ns[-1] - Ns[-2]) / N_STEPS
n_star  = t0_step / slope          # crossover N* = t0 / slope

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(Ns, ts * 1e3, 'o-', label='warm median')
ax.loglog(Ns, (t0_step + slope * Ns) * N_STEPS * 1e3, '--', color='#888',
          label=r'model $n_\mathrm{steps}\,(t_0 + N/B)$')
ax.axvline(n_star, color='#c33', ls=':', label=f'crossover N* ~ {n_star:,.0f}')
ax.set(xlabel='N (cells)', ylabel=f'time per {N_STEPS}-step run [ms]',
       title='Fixed per-step cost dominates until N is large enough')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f't0 = {t0_step*1e6:.1f} us/step    crossover N* ~ {n_star:,.0f} cells')


### <a id="sec7"></a>7. Cache residency

Throughput in the sweep above rises with `N`, then flattens. What sets the ceiling?

Query the L2 size, read XLA's working-set footprint from
`compiled.memory_analysis()`, and plot throughput against it.

---

**Quick Docs**

- `jax.jit(fn).lower(x).compile().memory_analysis()`: static buffer sizes of the
  compiled program; `temp_size_in_bytes` is the intermediate working set.
- `cupy.cuda.runtime.getDeviceProperties(0)['l2CacheSize']`: this GPU's L2 size.

In [ ]:
import cupy

# Query L2 size. Sweep grid sizes with increasing XLA buffer footprint timing a fixed-length scan on each.
L2_BYTES = int(cupy.cuda.runtime.getDeviceProperties(0)['l2CacheSize'])
EC_STEPS = 400
scan_solve = jax.jit(lambda s: lax.scan(step_jax, s, jnp.arange(EC_STEPS))[0])

# TODO: return (footprint_MB, throughput_Gcells_s) for grid size N_local.
#   - footprint: inferrable by calling memory_analysis() on function returned by compile()
#   - throughput: use timed_run with N_local * EC_STEPS, "min_s" - best of the repeats
def measure(N_local):
    ...

# Footprint is ~24 bytes/cell of XLA buffers. The sweep spans ~0.5x - ~6x the L2 size.
L2_SWEEP = [int(m * L2_BYTES / 24) for m in (0.5, 0.75, 1, 1.5, 2, 3, 4.5, 6)]
foot_mb, gcs = [], []
for N_local in L2_SWEEP:
    foot, gcell = measure(N_local)
    foot_mb.append(foot)
    gcs.append(gcell)

In [ ]:
L2_MB = swe_core.to_mib(L2_BYTES)
pk = int(np.argmax(gcs))
peak, floor = gcs[pk], gcs[-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(foot_mb, gcs, 'o-', color='#27a')
ax.axvline(L2_MB, color='#c33', ls='--', label=f'L2 = {L2_MB:.0f} MB')
ax.axvspan(foot_mb[0], L2_MB, alpha=0.05, color='green')
ax.axvspan(L2_MB, foot_mb[-1], alpha=0.06, color='#c33')
ax.set_ylim(0, peak * 1.28)
ax.annotate(f'{peak:.0f} Gcells/s', xy=(foot_mb[pk], peak), xytext=(0, 8),
            textcoords='offset points', ha='center', va='bottom', fontsize=9, color='#27a',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#27a', lw=0.8))
ax.annotate(f'~{floor:.0f} Gcells/s\nDRAM-bound, {peak/floor:.1f}x lower', xy=(foot_mb[-1], floor),
            xytext=(foot_mb[-1], floor + peak * 0.2), color='#c33', fontsize=9, ha='right')
ax.set_xlabel('working-set footprint [MB]')
ax.set_ylabel('throughput [Gcells/s]')
ax.set_title('Throughput peak')
ax.legend(loc='center right', fontsize=8)
plt.tight_layout(); plt.show()

# Device-only DRAM bandwidth
print(f'DRAM plateau: {gcs[-1]:.1f} Gcells/s -> {gcs[-1] * 16:.0f} GB/s of useful traffic')

**EXTRA CREDIT: inspecting intermediate states**

`lax.scan` runs the whole loop as one compiled program, so there is no Python
loop to `print` from. Inside it `h` is a placeholder (a *tracer*), not real
numbers, so `float(h[i])` fails. To watch the trajectory, have the scan body
return a value each step, and `scan` collects them into an array you get back
at the end.

---

**Quick Docs**

- `lax.scan(body, init, xs)`: the second element of `body`'s return is
  stacked across steps and returned as the scan's second output.

In [ ]:
# EXTRA CREDIT: collect the trajectory by returning h from the scan body.
state0 = (jnp.asarray(swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)[0], jnp.float32),
          jnp.zeros(N + 2, jnp.float32))

# A mid-loop h is a tracer, so float(h[i]) inside the scan body raises. The fix
# is to return a per-step diagnostic from the body, which scan stacks into ys.
# TODO: run a scan whose body returns (new_state, per-step total water volume,
# jnp.sum(new_state[0][1:-1])), then block_until_ready and confirm you get one
# sample per step.
...

**Float64 for the synthesis comparison.** Notebook 14 compares every tool at
matched precision. The cell below runs the same scanned solve in float64 and
records rates across the shared size range.

In [ ]:
%%writefile swe_jax_step.py
# The float64 step lives in a file so this notebook and the profiler's
# subprocess run the same code. Re-run this cell and the next one after
# any edit here, or the imported copy stays stale.
from functools import partial

import jax
import jax.numpy as jnp
import swe_core

jax.config.update('jax_enable_x64', True)
G = 9.81


@jax.jit
def step64(state, _, dx_, dt_):
    h, hu = state
    h  = h.at[0].set(h[1]).at[-1].set(h[-2])
    hu = hu.at[0].set(-hu[1]).at[-1].set(-hu[-2])
    hL, hR = h[:-1], h[1:]
    huL, huR = hu[:-1], hu[1:]
    hsL = jnp.maximum(hL, swe_core.DRY_TOL)
    hsR = jnp.maximum(hR, swe_core.DRY_TOL)
    uL, uR = huL / hsL, huR / hsR
    cL, cR = jnp.sqrt(G * hsL), jnp.sqrt(G * hsR)
    a = jnp.maximum(jnp.abs(uL) + cL, jnp.abs(uR) + cR)
    F_h  = 0.5 * (huL + huR) - 0.5 * a * (hR - hL)
    F_hu = 0.5 * (huL*uL + 0.5*G*hL*hL + huR*uR + 0.5*G*hR*hR) - 0.5 * a * (huR - huL)
    h_new  = h.at[1:-1].set(h[1:-1]  - (dt_/dx_) * (F_h[1:]  - F_h[:-1]))
    hu_new = hu.at[1:-1].set(hu[1:-1] - (dt_/dx_) * (F_hu[1:] - F_hu[:-1]))
    return (h_new, hu_new), None


# Wrapping the scan in jit traces the body once. Calling lax.scan directly
# re-traces it on every call.
@partial(jax.jit, static_argnames=('n_steps',))
def solve64(state, dx_, dt_, n_steps):
    final, _ = jax.lax.scan(lambda s, x: step64(s, x, dx_, dt_), state,
                            jnp.arange(n_steps))
    return final[0]


In [ ]:
# Float64 rates across the shared size range, for the synthesis notebook (14).
jax.config.update('jax_enable_x64', True)

# Written by the cell above; reload so an edit there takes effect here.
import importlib
import swe_jax_step
solve64 = importlib.reload(swe_jax_step).solve64

_setup64 = swe_core.by_size(lambda n: tuple(
    jnp.asarray(a) for a in swe_core.bump_ic(n, L=L, h0=H0, amplitude=AMP, sigma=SIG)))
_setup32 = swe_core.by_size(lambda n: tuple(
    jnp.asarray(a, jnp.float32)
    for a in swe_core.bump_ic(n, L=L, h0=H0, amplitude=AMP, sigma=SIG)))


def run_jax64(n_cells, n_steps):
    dx_ = L / n_cells
    dt_ = swe_core.fixed_dt(H0 + AMP, dx_, cfl=CFL, g=G)
    return jax.block_until_ready(solve64(_setup64(n_cells), dx_, dt_, n_steps))

warm64 = swe_core.timed_run(run_jax64, N, N_STEPS, label='09_jax_fp64')
diff64 = swe_core.max_diff(h_ref, np.asarray(run_jax64(N, N_STEPS)))
swe_core.report_and_verify(warm64, diff64, tol=1e-12, n=N, steps=N_STEPS)
swe_core.save_timing(warm64, grid_str=f'N={N}', tool='jax_fp64', hardware='gpu',
                     dtype='float64', steps=N_STEPS, max_diff_vs_numpy=diff64)
swe_core.save_sweep('09_jax_fp64', run_jax64)

# Matched fp32 point at the largest size: the DRAM-resident precision gap.
def run_jax32(n_cells, n_steps):
    dx_ = L / n_cells
    dt_ = swe_core.fixed_dt(H0 + AMP, dx_, cfl=CFL, g=G)
    return jax.block_until_ready(solve64(_setup32(n_cells), dx_, dt_, n_steps))

BN, BSTEPS = swe_core.SWEEP_SIZES[-1]
t64 = swe_core.timed_run(run_jax64, BN, BSTEPS, warmup=1, repeats=3)
t32 = swe_core.timed_run(run_jax32, BN, BSTEPS, warmup=1, repeats=3)
print(f"DRAM-resident fp64/fp32: {t64['median_s'] / t32['median_s']:.1f}x")

**Recap.**

- We expressed our computation as a **pure function** of `(h, hu)` and let
  `@jax.jit + lax.scan` fuse the whole time loop into one device program.
- **Shape-rigidity**: changing `N` is not free.
- Below the crossover **N\*** (Sec. 6) a fixed per-step cost dominates:
  the grid is too small for the GPU to matter. Above it, time scales with `N`.
- JAX is float32-first: the docs call single precision 'the desired
  behavior for many machine-learning applications'; float64 is opt-in
  (`jax_enable_x64`) and pays the throughput hit measured above.

Next: `10__swe__pyomp.ipynb` uses OpenMP's standard pragma-based API for shared-memory parallelism from Python.